# Week 1 — Dataset Feasibility Report
**Project:** Drug Side Effect Predictor  
**Goal:** Answer the 10 feasibility questions from the project proposal before any modeling begins.

We are loading the raw SIDER files and inspecting them. No cleaning, no modeling — just understanding what we have.

## Setup

In [2]:
import pandas as pd

# Path to your raw data folder
RAW = r"D:\Experimental Stuff\Data Science\Drug-Side-Effect-Predictor\data\raw\\"

print("Libraries loaded.")

Libraries loaded.


---
## Q1 — What files does SIDER contain?
Already answered from the format description. Files we will use:
- `drug_names.tsv` — drug name to STITCH ID mapping
- `drug_atc.tsv` — drug to ATC code (therapeutic class)
- `meddra_all_se.tsv` — drug to side effect associations (our label source)
- `meddra_all_indications.tsv` — drug to indication (textual feature)
- `meddra.tsv` — UMLS to MedDRA term mapping (for standardizing side effect names)

Files we are skipping for v1:
- `meddra_all_label_se.tsv` (533 MB, redundant for our purposes)
- `meddra_all_label_indications.tsv` (redundant)
- `meddra_freq.tsv` (optional, may use later)

---
## Q2 — What does each file represent?
Documented above. We will confirm structure by loading each file and checking its columns and first few rows.

---
## Q3 — What is the drug identifier?

In [3]:
# Load drug_names.tsv — no header in SIDER files, so we name columns manually
drug_names = pd.read_csv(
    RAW + "drug_names.tsv",
    sep="\t",
    header=None,
    names=["stitch_id_flat", "drug_name"]
)

print("Shape:", drug_names.shape)
print()
drug_names.head(10)

Shape: (1430, 2)



,stitch_id_flat,drug_name
0,CID100000085,carnitine
1,CID100000119,gamma-aminobutyric
2,CID100000137,5-aminolevulinic
3,CID100000143,leucovorin
4,CID100000146,5-methyltetrahydrofolate
5,CID100000158,PGE2
6,CID100000159,prostacyclin
7,CID100000160,prostaglandin
8,CID100000175,acetate
9,CID100000187,acetylcholine


**What to look for:** The `stitch_id_flat` column is the drug identifier used across all SIDER files. It should look like `CID000002082`. Note whether it has leading zeros — these matter for joining files later.

In [4]:
# Check for duplicate drug names or IDs
print("Unique STITCH IDs:", drug_names["stitch_id_flat"].nunique())
print("Unique drug names:", drug_names["drug_name"].nunique())
print("Total rows:", len(drug_names))
print()
# Are there drugs with multiple names, or names with multiple IDs?
print("IDs with more than one name:", (drug_names.groupby("stitch_id_flat")["drug_name"].count() > 1).sum())

Unique STITCH IDs: 1430
Unique drug names: 1347
Total rows: 1430

IDs with more than one name: 0


---
## Q4 — What is the side effect identifier?

In [5]:
# Load meddra.tsv — the side effect term reference table
meddra = pd.read_csv(
    RAW + "meddra.tsv",
    sep="\t",
    header=None,
    names=["umls_id", "meddra_id", "term_type", "side_effect_name"]
)

print("Shape:", meddra.shape)
print()
meddra.head(10)

Shape: (95912, 4)



,umls_id,meddra_id,term_type,side_effect_name
0,C0000727,LT,10000647,Acute abdomen
1,C0000727,PT,10000647,Acute abdomen
2,C0000727,LT,10042784,Syndrome abdominal acute
3,C0000727,LT,10000096,Abdominal syndrome acute
4,C0000729,LT,10000057,Abdominal cramps
5,C0000729,LT,10000056,Abdominal cramp
6,C0000729,LT,10011286,Cramp abdominal
7,C0000729,LT,10000058,Abdominal crampy pains
8,C0000729,LT,10018727,Griping abdominal
9,C0000729,LT,10018726,Griping abdomen


In [6]:
# What term types exist?
print("Term types:")
print(meddra["term_type"].value_counts())
print()
# We will use PT (Preferred Term) only — filters out redundant LLT synonyms
meddra_pt = meddra[meddra["term_type"] == "PT"]
print("Unique PT side effects:", meddra_pt["side_effect_name"].nunique())

Term types:
term_type
10018865    4
10018851    4
10018029    4
10002084    4
10028941    4
           ..
10073967    1
10073984    1
10074155    1
10074210    1
10074226    1
Name: count, Length: 74359, dtype: int64

Unique PT side effects: 0


**What to look for:** PT = Preferred Term is the standardized label. LLT = Lowest Level Term are synonyms that map to the same PT. We want PT only to avoid counting the same side effect multiple times under different names.

---
## Q5 — How many unique drugs exist?

In [7]:
# Already partially answered above from drug_names
# Now also check drug_atc.tsv to see if the drug count is consistent
drug_atc = pd.read_csv(
    RAW + "drug_atc.tsv",
    sep="\t",
    header=None,
    names=["stitch_id_flat", "atc_code"]
)

print("drug_atc shape:", drug_atc.shape)
print("Unique drugs in drug_atc:", drug_atc["stitch_id_flat"].nunique())
print()
drug_atc.head(10)

drug_atc shape: (1560, 2)
Unique drugs in drug_atc: 1118



,stitch_id_flat,atc_code
0,CID100000085,A16AA01
1,CID100000119,L03AA03
2,CID100000119,N03AG03
3,CID100000137,L01XD04
4,CID100000143,V03AF03
5,CID100000143,V03AF04
6,CID100000143,V03AF06
7,CID100000158,G02AD02
8,CID100000159,B01AC09
9,CID100000160,G02AD01


In [8]:
# How many drugs have ATC codes vs total drugs?
drugs_with_names = set(drug_names["stitch_id_flat"])
drugs_with_atc = set(drug_atc["stitch_id_flat"])

print("Drugs with names:", len(drugs_with_names))
print("Drugs with ATC codes:", len(drugs_with_atc))
print("Drugs with BOTH:", len(drugs_with_names & drugs_with_atc))
print("Drugs with names but NO ATC code:", len(drugs_with_names - drugs_with_atc))

Drugs with names: 1430
Drugs with ATC codes: 1118
Drugs with BOTH: 1118
Drugs with names but NO ATC code: 312


**What to look for:** ATC code is your main categorical feature (therapeutic class). If many drugs lack an ATC code, that limits your feature set. Note the gap — it will matter during modeling.

---
## Q6 — How many unique side effects exist?

In [9]:
# Load meddra_all_se.tsv — the main drug-side effect association file
# This file has 10 columns per the format description
se = pd.read_csv(
    RAW + "meddra_all_se.tsv",
    sep="\t",
    header=None,
    names=[
        "stitch_id_flat",
        "stitch_id_stereo",
        "umls_id_label",
        "meddra_type",
        "umls_id_meddra",
        "side_effect_name"
    ]
)

print("Shape:", se.shape)
print()
se.head(10)

Shape: (309849, 6)



,stitch_id_flat,stitch_id_stereo,umls_id_label,meddra_type,umls_id_meddra,side_effect_name
0,CID100000085,CID000010917,C0000729,LLT,C0000729,Abdominal cramps
1,CID100000085,CID000010917,C0000729,PT,C0000737,Abdominal pain
2,CID100000085,CID000010917,C0000737,LLT,C0000737,Abdominal pain
3,CID100000085,CID000010917,C0000737,PT,C0687713,Gastrointestinal pain
4,CID100000085,CID000010917,C0000737,PT,C0000737,Abdominal pain
5,CID100000085,CID000010917,C0002418,LLT,C0002418,Amblyopia
6,CID100000085,CID000010917,C0002418,PT,C0002418,Amblyopia
7,CID100000085,CID000010917,C0002871,LLT,C0002871,Anaemia
8,CID100000085,CID000010917,C0002871,PT,C0002871,Anaemia
9,CID100000085,CID000010917,C0003123,LLT,C0003123,Anorexia


In [10]:
# Filter for PT only (same reasoning as before)
se_pt = se[se["meddra_type"] == "PT"]

print("Total rows (all term types):", len(se))
print("Rows with PT only:", len(se_pt))
print()
print("Unique side effects (PT):", se_pt["side_effect_name"].nunique())
print("Unique side effects (all):", se["side_effect_name"].nunique())

Total rows (all term types): 309849
Rows with PT only: 163206

Unique side effects (PT): 4251
Unique side effects (all): 6123


---
## Q7 — How many drug–side effect associations exist?

In [11]:
print("Total drug-side effect associations (PT only):", len(se_pt))
print("Unique drugs in association file:", se_pt["stitch_id_flat"].nunique())
print()
# Average side effects per drug
avg_se = se_pt.groupby("stitch_id_flat")["side_effect_name"].count()
print("Avg side effects per drug:", round(avg_se.mean(), 1))
print("Min:", avg_se.min(), "  Max:", avg_se.max())
print()
# Most common side effects
print("Top 20 most common side effects:")
print(se_pt["side_effect_name"].value_counts().head(20))

Total drug-side effect associations (PT only): 163206
Unique drugs in association file: 1430

Avg side effects per drug: 114.1
Min: 1   Max: 1136

Top 20 most common side effects:
side_effect_name
Dizziness                     1758
Asthenia                      1513
Dermatitis                    1433
Nausea                        1302
Rash                          1250
Musculoskeletal discomfort    1242
Headache                      1219
Vomiting                      1175
Diarrhoea                     1148
Feeling abnormal              1133
Pruritus                      1027
Abdominal pain                1009
Hypersensitivity               984
Decreased appetite             955
Pain                           945
Gastrointestinal disorder      928
Body temperature increased     892
Dyspepsia                      890
Urticaria                      888
Gastrointestinal pain          877
Name: count, dtype: int64


---
## Q8 — What drug-level attributes are actually available?

In [12]:
# Load meddra_all_indications.tsv — drug indication (what condition the drug treats)
indications = pd.read_csv(
    RAW + "meddra_all_indications.tsv",
    sep="\t",
    header=None,
    names=[
        "stitch_id_flat",
        "umls_id_label",
        "method",
        "concept_name",
        "meddra_type",
        "umls_id_meddra",
        "meddra_concept_name"
    ]
)

print("Shape:", indications.shape)
print()
indications.head(10)

Shape: (30835, 7)



,stitch_id_flat,umls_id_label,method,concept_name,meddra_type,umls_id_meddra,meddra_concept_name
0,CID100000085,C0015544,text_mention,Failure to Thrive,LLT,C0015544,Failure to thrive
1,CID100000085,C0015544,text_mention,Failure to Thrive,PT,C0015544,Failure to thrive
2,CID100000085,C0020615,text_mention,Hypoglycemia,LLT,C0020615,Hypoglycaemia
3,CID100000085,C0020615,text_mention,Hypoglycemia,PT,C0020615,Hypoglycaemia
4,CID100000085,C0022661,NLP_indication,"Kidney Failure, Chronic",LLT,C0022661,Renal failure chronic
5,CID100000085,C0022661,NLP_indication,"Kidney Failure, Chronic",PT,C0022661,Renal failure chronic
6,CID100000085,C0025521,NLP_indication,Inborn Errors of Metabolism,LLT,C0025521,Inborn error of metabolism
7,CID100000085,C0025521,NLP_indication,Inborn Errors of Metabolism,PT,C0025521,Inborn error of metabolism
8,CID100000085,C0026827,text_mention,Muscle hypotonia,LLT,C0026827,Hypotonia
9,CID100000085,C0026827,text_mention,Muscle hypotonia,PT,C0026827,Hypotonia


In [13]:
# Filter PT only for indications too
ind_pt = indications[indications["meddra_type"] == "PT"]

print("Unique drugs with indications:", ind_pt["stitch_id_flat"].nunique())
print("Unique indications:", ind_pt["meddra_concept_name"].nunique())
print()
# Summarise what attributes we actually have per drug
print("--- Attribute Summary ---")
print("Drug names available for:", drug_names["stitch_id_flat"].nunique(), "drugs")
print("ATC codes available for: ", drug_atc["stitch_id_flat"].nunique(), "drugs")
print("Indications available for:", ind_pt["stitch_id_flat"].nunique(), "drugs")
print("Side effect labels for:  ", se_pt["stitch_id_flat"].nunique(), "drugs")

Unique drugs with indications: 1437
Unique indications: 2213

--- Attribute Summary ---
Drug names available for: 1430 drugs
ATC codes available for:  1118 drugs
Indications available for: 1437 drugs
Side effect labels for:   1430 drugs


---
## Q9 — Can the drug IDs be joined across files?

In [14]:
# Check how many drugs appear in ALL four sources
drugs_se = set(se_pt["stitch_id_flat"])
drugs_ind = set(ind_pt["stitch_id_flat"])
drugs_names = set(drug_names["stitch_id_flat"])
drugs_atc = set(drug_atc["stitch_id_flat"])

core = drugs_se & drugs_names  # drugs with labels AND a name (minimum viable)
full = drugs_se & drugs_names & drugs_atc & drugs_ind  # drugs with everything

print("Drugs with side effects + name (minimum viable):", len(core))
print("Drugs with ALL four attributes:               ", len(full))
print()
print("Drugs with side effects but no ATC code:", len(drugs_se - drugs_atc))
print("Drugs with side effects but no indication:", len(drugs_se - drugs_ind))

Drugs with side effects + name (minimum viable): 1430
Drugs with ALL four attributes:                1076

Drugs with side effects but no ATC code: 312
Drugs with side effects but no indication: 70


---
## Q10 — Can a valid drug-level modeling table be constructed?

In [15]:
# Build a minimal candidate modeling table: one row per drug
# with at least a name, one attribute, and at least one side effect label

# Aggregate indications per drug into a single text string (for TF-IDF later)
ind_agg = ind_pt.groupby("stitch_id_flat")["meddra_concept_name"].apply(lambda x: " ".join(x)).reset_index()
ind_agg.columns = ["stitch_id_flat", "indications_text"]

# Aggregate ATC codes per drug (a drug can have multiple ATC codes)
atc_agg = drug_atc.groupby("stitch_id_flat")["atc_code"].apply(lambda x: "|".join(x)).reset_index()
atc_agg.columns = ["stitch_id_flat", "atc_codes"]

# Build the drug table
drug_table = drug_names.merge(atc_agg, on="stitch_id_flat", how="left")
drug_table = drug_table.merge(ind_agg, on="stitch_id_flat", how="left")

# Keep only drugs that have at least one side effect label
drug_table = drug_table[drug_table["stitch_id_flat"].isin(drugs_se)]

print("Candidate modeling table shape:", drug_table.shape)
print()
print("Missing ATC codes:", drug_table["atc_codes"].isna().sum())
print("Missing indications:", drug_table["indications_text"].isna().sum())
print()
drug_table.head(10)

Candidate modeling table shape: (1430, 4)

Missing ATC codes: 312
Missing indications: 70



,stitch_id_flat,drug_name,atc_codes,indications_text
0,CID100000085,carnitine,A16AA01,Failure to thrive Hypoglycaemia Renal failure ...
1,CID100000119,gamma-aminobutyric,L03AA03|N03AG03,Hypogammaglobulinaemia Varicella Immunodeficie...
2,CID100000137,5-aminolevulinic,L01XD04,Actinic keratosis Seborrhoeic keratosis Porphy...
3,CID100000143,leucovorin,V03AF03|V03AF04|V03AF06,Anaemia Anaemia megaloblastic Pernicious anaem...
4,CID100000146,5-methyltetrahydrofolate,NaN,NaN
5,CID100000158,PGE2,G02AD02,Abdominal pain Abortion missed Cardiovascular ...
6,CID100000159,prostacyclin,B01AC09,Connective tissue disorder Scleroderma Hyperte...
7,CID100000160,prostaglandin,G02AD01,NaN
8,CID100000175,acetate,G01AD02|S02AA10,Infection Hyponatraemia
9,CID100000187,acetylcholine,S01EB09,Miosis


In [16]:
# Label space: how many side effects appear in at least N drugs?
# This is critical — rare labels are a major risk (Section 15 of proposal)
se_counts = se_pt["side_effect_name"].value_counts()

for threshold in [1, 5, 10, 25, 50, 100]:
    count = (se_counts >= threshold).sum()
    print(f"Side effects appearing in >= {threshold:3d} drugs: {count}")

Side effects appearing in >=   1 drugs: 4251
Side effects appearing in >=   5 drugs: 2061
Side effects appearing in >=  10 drugs: 1471
Side effects appearing in >=  25 drugs: 919
Side effects appearing in >=  50 drugs: 606
Side effects appearing in >= 100 drugs: 391


---
## Summary

Fill this in after running all cells above:

| Question | Answer |
|---|---|
| Q1: Files in SIDER | 7 usable files (5 used in v1) |
| Q2: What each file represents | Documented in cells above |
| Q3: Drug identifier | `stitch_id_flat` (CID... format) |
| Q4: Side effect identifier | UMLS concept ID, standardized to MedDRA PT name |
| Q5: Unique drugs | Unique drugs in drug_atc: 1118, Drugs with names: 1430, Drugs with ATC codes: 1118, Drugs with BOTH: 1118, Drugs with names but NO ATC code: 312 |
| Q6: Unique side effects (PT) | Total rows (all term types): 309849, Rows with PT only: 163206, Unique side effects (PT): 4251, Unique side effects (all): 6123 |
| Q7: Total associations (PT) | Total drug-side effect associations (PT only): 163206, Unique drugs in association file: 1430 |
| Q8: Drug attributes available | Drug names available for: 1430 drugs, ATC codes available for:  1118 drugs, Indications available for: 1437 drugs, Side effect labels for: 1430 drugs |
| Q9: Can files be joined? | Drugs with side effects + name (minimum viable): 1430, Drugs with ALL four attributes: 1076, Drugs with side effects but no ATC code: 312, Drugs with side effects but no indication: 70 |
| Q10: Can modeling table be built? | Candidate modeling table shape: (1430, 4), Missing ATC codes: 312, Missing indications: 70, Side effects appearing in >=   1 drugs: 4251, Side effects appearing in >=   5 drugs: 2061, Side effects appearing in >=  10 drugs: 1471, Side effects appearing in >=  25 drugs: 919, Side effects appearing in >=  50 drugs: 606, Side effects appearing in >= 100 drugs: 391 |

**Decision (per Section 7 of proposal):**  
*Since, the SIDER data turned out to be pretty good we'll proceed with SIDER*